<!-- launch-badges -->
<a href="https://colab.research.google.com/github/laban254/ml-for-infrastructure/blob/main/01_foundations/numpy/numpy.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
&nbsp;
<a href="https://mybinder.org/v2/gh/laban254/ml-for-infrastructure/main?urlpath=lab/tree/01_foundations/numpy/numpy.ipynb" target="_blank"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"/></a>

> ▶️ **Run this notebook live** — no install needed. Click a badge above to open it in a free cloud runtime.

# Multi-Dimensional Arrays with NumPy

## Context
As an SRE, you often deal with massive streams of numerical data: CPU utilization percentages, network throughput in bytes, or memory consumption across hundreds of containers. Python lists are far too slow and memory-inefficient to handle this at scale.

## Objectives
- Understand NumPy's core object: the `ndarray` (N-dimensional array).
- Learn how to perform vectorized operations on arrays without slow `for` loops.
- Apply basic statistical functions to uncover anomalies in synthetic infrastructure metrics.

## Expected Outcome
- The ability to quickly analyze thousands of datapoints, calculate baselines, and detect spikes using pure mathematics.

In [1]:
import numpy as np

### 1. Creating Arrays (Simulating Metrics)
An `ndarray` allows you to store and manipulate large datasets efficiently. Let's create some arrays simulating CPU load percentages across different servers.

In [2]:
# 1D Array: CPU load recorded every minute for 1 server
cpu_server_1 = np.array([45, 48, 52, 50, 47])
print("1D Array (Server 1):", cpu_server_1)

# 2D Array (Matrix): CPU load for 3 separate servers over 4 minutes
cpu_cluster = np.array([
    [45, 48, 52, 50], # Server A
    [20, 22, 21, 23], # Server B
    [85, 88, 90, 89]  # Server C (Spiking!)
])
print("\n2D Array (Cluster Metrics):\n", cpu_cluster)

1D Array (Server 1): [45 48 52 50 47]

2D Array (Cluster Metrics):
 [[45 48 52 50]
 [20 22 21 23]
 [85 88 90 89]]


### 2. Inspecting Array Properties
When you load a dataset from a log file, the first step is understanding its dimensions.

In [3]:
print("Shape of cpu_cluster:", cpu_cluster.shape)  # Output: (3 servers, 4 minutes)
print("Number of dimensions:", cpu_cluster.ndim)
print("Total data points:", cpu_cluster.size)

Shape of cpu_cluster: (3, 4)
Number of dimensions: 2
Total data points: 12


### 3. Generating Data
Often you need to generate placeholder metrics or baseline limits.

In [4]:
# Create an array representing a 0% load baseline for 5 servers
zeros_array = np.zeros(5)
print("Zero Baseline:\n", zeros_array)

# Create an array representing a max capacity limit (100%)
max_capacity = np.full((3, 4), 100)
print("\nMax Capacity Matrix:\n", max_capacity)

# Generate a sequence of timestamps (e.g., 0 to 60 seconds, step 10)
timestamps = np.arange(0, 60, 10)
print("\nTimestamps:", timestamps)

Zero Baseline:
 [0. 0. 0. 0. 0.]

Max Capacity Matrix:
 [[100 100 100 100]
 [100 100 100 100]
 [100 100 100 100]]

Timestamps: [ 0 10 20 30 40 50]


### 4. Indexing and Slicing (Isolating Servers)
If Server C is throwing alerts, we need to extract only its data from the matrix.

In [5]:
# Accessing specific elements
print("Load for Server B at Minute 2:", cpu_cluster[1, 1]) 

# Slicing entire rows/columns
print("All metrics for Server C:", cpu_cluster[2, :])
print("Metrics across all servers at Minute 3:", cpu_cluster[:, 2])

Load for Server B at Minute 2: 22
All metrics for Server C: [85 88 90 89]
Metrics across all servers at Minute 3: [52 21 90]


### 5. Element-wise Operations & Broadcasting
NumPy's greatest strength is performing mathematical operations on entire arrays at once, without `for` loops. This is called *vectorization*.

In [6]:
ram_used_gb = np.array([16, 32, 8])
ram_total_gb = np.array([64, 64, 16])

# Element-wise calculation to find utilization percentage
utilization = (ram_used_gb / ram_total_gb) * 100
print("RAM Utilization %:", utilization)

# Broadcasting: Subtracting a scalar from an array
# Imagine a new software update reduces RAM usage by 2GB across all servers
new_ram_used = ram_used_gb - 2
print("\nNew RAM Usage:", new_ram_used)

RAM Utilization %: [25. 50. 50.]

New RAM Usage: [14 30  6]


### 6. Statistical Operations (Finding Anomalies)
Using built-in stats functions allows you to immediately identify outliers in your cluster.

In [7]:
print("Matrix:\n", cpu_cluster)

# Global stats
print("\nGlobal Max CPU Load:", np.max(cpu_cluster))
print("Global Mean CPU Load:", np.mean(cpu_cluster))

# Axis-specific stats (0 = columns, 1 = rows)
# Calculate the mean load for each individual server
server_means = np.mean(cpu_cluster, axis=1)
print("\nMean Load per Server:", server_means)

# Calculate the 99th percentile across all data (crucial for SLAs)
p99 = np.percentile(cpu_cluster, 99)
print("99th Percentile CPU Load:", p99)

Matrix:
 [[45 48 52 50]
 [20 22 21 23]
 [85 88 90 89]]

Global Max CPU Load: 90
Global Mean CPU Load: 52.75

Mean Load per Server: [48.75 21.5  88.  ]
99th Percentile CPU Load: 89.89


### 7. Boolean Masks (Filtering Alerts)
We can use arrays of booleans to filter raw data and retrieve only the values that breach our thresholds.

In [8]:
# Find all instances where CPU load exceeded 80%
alert_mask = cpu_cluster > 80
print("Alert Mask:\n", alert_mask)

# Use the mask to extract those specific dangerous values
dangerous_values = cpu_cluster[alert_mask]
print("\nValues exceeding 80%:", dangerous_values)

Alert Mask:
 [[False False False False]
 [False False False False]
 [ True  True  True  True]]

Values exceeding 80%: [85 88 90 89]


---
## 🧪 Try it yourself: move the alert threshold

Drag the threshold and watch which CPU readings breach it. This is the raw mechanic behind every static Grafana alert rule.

> **Note:** Dragging the slider requires a live kernel (e.g. open this notebook in Colab or Binder via the badges at the top) — it won't be interactive on a static GitHub view.

In [9]:
from ipywidgets import interact, IntSlider

def show_alerts(threshold=80):
    mask = cpu_cluster > threshold
    print(f'Threshold {threshold}% -> {mask.sum()} breaching readings')
    print('Breaching values:', cpu_cluster[mask])

interact(show_alerts, threshold=IntSlider(min=20, max=100, step=5, value=80));

interactive(children=(IntSlider(value=80, description='threshold', min=20, step=5), Output()), _dom_classes=('…

## 📝 Exercise: per-server p99

Static thresholds are crude. Compute the **99th percentile** of each server's load (one value per row of `cpu_cluster`) and print which servers have a p99 above 80%.

```python
# your code here
# hint: np.percentile(..., axis=1)
```

<details><summary>💡 Reveal solution</summary>

```python
p99 = np.percentile(cpu_cluster, 99, axis=1)
for i, val in enumerate(p99):
    flag = '⚠️ HOT' if val > 80 else 'ok'
    print(f'Server {i}: p99 = {val:.1f}%  {flag}')
```
</details>